
# T3.3 EIT-003 Lifecycle Environmental Performance Model

READ.ME 

This notebook is the **shared foundation** for the EIT-003 pipeline. It contains everything that is independent of runtime event data and must be run once before EIT-001 or EIT-002 can be used.

- Environment setup (Brightway project initialisation ecoinvent and BONSAI database installation)
- Foreground model: REED foreground database imported from `REED_model_*.xlsx`
- Baseline LCA results 
  - KOMATSU results
  - WWR results
  - GOIMEK A1 / A2 / A3 results
- Analysis & visualisation (Contribution analysis, contribution plots)
- Reusable LCA functions (`lca_score`, `run_contribution_analysis`, `lcia_for_activity_with_scenario`)

Run this notebook **once** to set up the Brightway project.

> EIT-001 and EIT-002 both assume this project is initialised and call into it.

### 1. Importers

#### 1.1 Brightway

In [ ]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import CONFIG
import bw_processing as bwp
from bw2io.strategies.generic import set_code_by_activity_hash


import matplotlib.pyplot as plt
import d3graph
import polyviz

from datetime import date

In [ ]:
config = CONFIG

In [ ]:
bd.projects.set_current(config.PROJECT_NAME)

In [ ]:
#bi.remote.install_project(project_key="ecoinvent-3.10-biosphere", overwrite_existing=True)

#### 1.2 BONSAI and REED database

In [ ]:
import import_db

In [ ]:
import_db.main(config.BONSAI_PATH)

In [ ]:
bonsai = bd.Database('BONSAI_V2.1.6')
bonsai_bio = bd.Database('BONSAI_V2.1.6 biosphere')
ecoinvent_bio = bd.Database('ecoinvent-3.10-biosphere')

In [ ]:
from bw2io.importers import ExcelImporter
from reed.utils import Config

importer  = ExcelImporter(config.REED_PATH/'Model'/'REED_model_20260506.xlsx',)
importer.apply_strategies()
importer.match_database(fields=['name','unit','location'])
importer.match_database(db_name=bonsai.name,fields=['name','unit','location','activity type'])
importer.match_database(db_name=bd.config.biosphere,fields=['name','categories','unit'])


In [ ]:
importer.statistics()

In [ ]:
importer.write_database()
reed = bd.Database('reed')

In [ ]:
data_reed = [activity.as_dict() for activity in reed]
df_reed = pd.DataFrame(data_reed)


In [ ]:
from IPython.display import display
display(df_reed[['name', 'comment', 'location', 'unit', 'database']])

In [ ]:
bonsai = bd.Database('BONSAI_V2.1.6')
bonsai_bio = bd.Database('BONSAI_V2.1.6 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')
reed = bd.Database('reed')

In [ ]:
bd.databases

# Supplementary Material

### 2. Results without Footprint Monitoring

In [ ]:
reed.search('goimek')

In [ ]:
komatsu_processes = {
    "KOMATSU total": reed.get(name='KOMATSU total', location="GB", unit='unit'),
    "KOMATSU A1: raw materials": reed.get(name='KOMATSU A1: raw materials', location="GB", unit='unit'),
    "KOMATSU A2: transport": reed.get(name='KOMATSU A2: transport', location="GB", unit='unit'),
    "KOMATSU A3: machining": reed.get(name='KOMATSU A3: machining', location="GB", unit='unit'),
    
}

In [ ]:
wwr_processes = {
    "WWR total": reed.get(name='WWR total', location="DE", unit='unit'),
    "WWR A1: material inputs": reed.get(name='WWR A1: material inputs', location="DE", unit='unit'),
    "WWR A2: transport": reed.get(name='WWR A2: transport', location="DE", unit='unit'),
    "WWR A3: machining": reed.get(name='WWR A3: machining', location="DE", unit='unit'),
}

In [ ]:
goimek_processes = {
    "GOIMEK total": reed.get(name='GOIMEK total', location="ES", unit='unit'),
    "GOIMEK A1: material input": reed.get(name='GOIMEK A1: material input', location="ES", unit='unit'),
    "GOIMEK A2: transport": reed.get(name='GOIMEK A2: transport', location="ES", unit='unit'),
    "GOIMEK A3: machining": reed.get(name='GOIMEK A3: machining', location="ES", unit='unit'),
}

In [ ]:
from reed.LCA_functions import (
    lca_score,
    run_contribution_analysis,
)

In [ ]:
df_komatsu_results = lca_score(komatsu_processes, config.lcia_gwp100)
display(df_komatsu_results)


In [ ]:
df_komatsu_results

In [ ]:
df_wwr_results = lca_score(wwr_processes, config.lcia_gwp100)
display(df_wwr_results)


In [ ]:
df_goimek_results = lca_score(goimek_processes, config.lcia_gwp100)
display(df_goimek_results)


In [ ]:
df_results_all = pd.concat(
    [df_komatsu_results, df_wwr_results, df_goimek_results],
    ignore_index=True
)
df_results_all

### 3. Plot Life Cycle Stages

In [ ]:
df_plot = df_komatsu_results.set_index("Process")

# Remove the row
df_plot = df_plot.drop("KOMATSU total", errors="ignore")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Life Cycle Phase in DPP")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
df_plot = df_wwr_results.set_index("Process")

# Remove the row
df_plot = df_plot.drop("WWR total", errors="ignore")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Life Cycle Phase in DPP")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
df_plot = df_goimek_results.set_index("Process")

# Remove the row
df_plot = df_plot.drop("GOIMEK total", errors="ignore")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Life Cycle Phase in DPP")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

### 4. Plot contribution analysis

In [ ]:
komatsu_ca = {}

for key, demand in komatsu_processes.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.001
    )
    komatsu_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )

In [ ]:
wwr_ca = {}

for key, demand in wwr_processes.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.001
    )
    wwr_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )

In [ ]:
goimek_ca = {}

for key, demand in goimek_processes.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.001
    )
    goimek_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )

In [ ]:
wwr_total = reed.get(name="WWR total", location="DE", unit="unit")

In [ ]:
bwa.print_recursive_calculation(wwr_total,
lcia_method=config.lcia_gwp100,max_level=1,cutoff=0.001)